<a href="https://colab.research.google.com/github/ChetanantKha/museum.db/blob/main/Copy_of_ARIMA_Workshop_Group4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: พยากรณ์ยอดขายร้านกาแฟด้วย ARIMA — กลุ่ม 4
**เวลา 60 นาที · งานกลุ่ม 3–4 คน · ไม่ต้องเขียนโค้ดเอง**

ร้านกาแฟ *Bean There สาขาเชียงใหม่* มียอดขายรายเดือน 6 ปี (2020–2025) หน่วยเป็น **พันแก้ว** และมีบันทึกว่าเดือนไหนจัดโปรโมชั่น
แต่ละกลุ่มได้ข้อมูลของสาขาต่างกัน ตัวเลขของกลุ่มจึงไม่ตรงกับกลุ่มอื่น
ภารกิจของกลุ่มคือหาโมเดลที่พยากรณ์ยอดขายได้แม่นที่สุด แล้วตอบคำถามผู้จัดการว่า **ธันวาคม 2026 ควรเตรียมวัตถุดิบสำหรับกี่แก้ว**

### วิธีใช้ notebook นี้
1. เปิดใน Google Colab แล้วกดเมนู **Runtime → Run all** (รันไทม์ → เรียกใช้ทั้งหมด) หนึ่งครั้งตอนเริ่ม
2. รันแต่ละเซลล์ด้วย **Shift + Enter** หรือกดปุ่ม ▶ ทางซ้าย
3. แก้ได้เฉพาะเซลล์ที่มีป้าย ✏️ โดยเลื่อนแถบเลื่อนหรือเลือกตัวเลือกทางขวา แล้วรันเซลล์นั้นใหม่
4. ตอบคำถาม 📝 ลงในใบงานกลุ่ม (หรือพิมพ์ในเซลล์คำตอบใต้คำถาม)

### แบ่งบทบาทในกลุ่ม
- **ผู้คุมเครื่อง** — คนกดรันและปรับค่า
- **นักวิเคราะห์** — อ่านกราฟและตัวเลข แล้วเสนอว่าจะลองค่าอะไรต่อ
- **ผู้จดบันทึก** — จดคำตอบและผลแต่ละรอบ
- **ผู้นำเสนอ** — สรุปผลให้ห้องฟัง 1 นาทีตอนท้าย

| ขั้น | กิจกรรม | นาที |
| --- | --- | --- |
| 0 | เตรียมเครื่องมือ | 5 |
| 1 | สำรวจข้อมูล | 8 |
| 2 | ตรวจความนิ่ง | 5 |
| 3 | แบ่ง train/test | 2 |
| 4 | โมเดล A: ARIMA | 8 |
| 5 | โมเดล B: SARIMA | 8 |
| 6 | โมเดล C: SARIMAX | 8 |
| 7 | ชาเลนจ์: แข่งลด MAPE | 8 |
| 8 | พยากรณ์ปี 2026 และนำเสนอ | 8 |

## ขั้นที่ 0 · เตรียมเครื่องมือ (5 นาที)
รันเซลล์ด้านล่างได้เลย ไม่ต้องอ่านโค้ด เซลล์นี้สร้างข้อมูลและฟังก์ชันช่วยที่ใช้ทั้ง workshop

In [ ]:
# @title ⚙️ กดรันเซลล์นี้ก่อน (ไม่ต้องแก้)
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ---- ฟอนต์ภาษาไทยสำหรับกราฟ ----
import os, glob, subprocess, urllib.request
from matplotlib import font_manager
def _setup_thai_font():
    try:
        path = "Sarabun-Regular.ttf"
        if not os.path.exists(path):
            urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf", path)
        font_manager.fontManager.addfont(path)
        return "Sarabun"
    except Exception:
        pass
    pattern = "/usr/share/fonts/**/tlwg/Loma*"
    files = glob.glob(pattern, recursive=True)
    if not files:
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-thai-tlwg"], capture_output=True)
        files = glob.glob(pattern, recursive=True)
    for f in files:
        font_manager.fontManager.addfont(f)
    return "Loma" if files else None
_font = _setup_thai_font()
if _font:
    plt.rcParams["font.family"] = _font
    plt.rcParams["axes.unicode_minus"] = False
else:
    print("⚠️ โหลดฟอนต์ภาษาไทยไม่ได้ ตัวอักษรไทยในกราฟอาจแสดงเป็นกล่องสี่เหลี่ยม")
TH_MONTHS = ["ม.ค.", "ก.พ.", "มี.ค.", "เม.ย.", "พ.ค.", "มิ.ย.", "ก.ค.", "ส.ค.", "ก.ย.", "ต.ค.", "พ.ย.", "ธ.ค."]
def th_month(dt):
    return f"{TH_MONTHS[dt.month - 1]} {dt.year}"

# ---- ข้อมูลจำลอง: ยอดขายรายเดือนร้าน Bean There (พันแก้ว) ----
GROUP, BRANCH = 4, "สาขาเชียงใหม่"
rng = np.random.default_rng(109)
idx = pd.date_range("2020-01-01", periods=72, freq="MS")
t = np.arange(72)
season_effect = {1: 7, 2: 4, 3: -1, 4: -3, 5: -4, 6: -5, 7: -4, 8: -4, 9: -3, 10: 1, 11: 5, 12: 9}
s = np.array([season_effect[d.month] for d in idx])
promo = np.zeros(72, int); promo[rng.choice(72, 14, replace=False)] = 1
sales = 30 + 0.4 * t + s + 6 * promo + rng.normal(0, 2.5, 72)
df = pd.DataFrame({"sales": sales.round(1), "promo": promo}, index=idx)
df.index.name = "month"

train, test = df.iloc[:60], df.iloc[60:]   # train = 2020–2024, test = 2025
results = []                                  # เก็บผลทุกโมเดลที่กลุ่มลอง

def run_model(name, p, d, q, P=0, D=0, Q=0, s=0, use_promo=False, show=True):
    """Fit บน train → พยากรณ์ 12 เดือนของ test → วัดผล"""
    try:
        exog_tr = train[["promo"]] if use_promo else None
        exog_te = test[["promo"]] if use_promo else None
        model = SARIMAX(train["sales"], exog=exog_tr, order=(p, d, q),
                        seasonal_order=(P, D, Q, s)).fit(disp=False)
        fc = model.get_forecast(12, exog=exog_te)
        pred, ci = fc.predicted_mean, fc.conf_int()
    except Exception as e:
        print("❌ โมเดลนี้รันไม่ได้ ลองลดค่า p, q, P หรือ Q ลง แล้วรันใหม่\n   (", str(e)[:80], ")")
        return None
    err = test["sales"] - pred
    mae = err.abs().mean(); rmse = np.sqrt((err ** 2).mean())
    mape = (err.abs() / test["sales"]).mean() * 100
    label = f"({p},{d},{q})" + (f"x({P},{D},{Q},{s})" if s else "") + (" + โปรโมชั่น" if use_promo else "")
    results.append({"ชื่อ": name, "โมเดล": label, "MAE": round(mae, 2),
                    "RMSE": round(rmse, 2), "MAPE %": round(mape, 2)})
    if show:
        fig, ax = plt.subplots()
        ax.plot(train.index, train["sales"], color="#9a9a9a", label="ข้อมูลฝึก (ค่าจริง)")
        ax.plot(test.index, test["sales"], color="black", lw=2, label="ข้อมูลทดสอบ (ค่าจริง)")
        ax.plot(pred.index, pred, color="black", lw=2, ls="--", label="ค่าพยากรณ์")
        ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="#cccccc", alpha=.5, label="ช่วงความเชื่อมั่น 95%")
        ax.set_title(f"{name}: SARIMAX{label}   |   MAPE = {mape:.1f}%")
        ax.set_ylabel("ยอดขาย (พันแก้ว)"); ax.legend(loc="upper left")
        plt.show()
        print(f"MAE  = {mae:.2f} พันแก้ว   (พลาดเฉลี่ยเดือนละ {mae*1000:,.0f} แก้ว)")
        print(f"RMSE = {rmse:.2f} พันแก้ว")
        print(f"MAPE = {mape:.2f} %")
    return model

def leaderboard():
    if not results:
        print("ยังไม่มีผล"); return
    return pd.DataFrame(results).sort_values("MAPE %").reset_index(drop=True)

print(f"✅ พร้อมแล้ว — กลุ่ม {GROUP} · Bean There {BRANCH} · มีข้อมูล", len(df), "เดือน ตั้งแต่", th_month(df.index[0]), "ถึง", th_month(df.index[-1]))
show = df.rename(columns={"sales": "ยอดขาย (พันแก้ว)", "promo": "โปรโมชั่น (1 = มี)"})
show.index = [th_month(x) for x in df.index]; show.index.name = "เดือน"
show.head(12)

## ขั้นที่ 1 · สำรวจข้อมูล (8 นาที)
ขั้นตอนแรกของการวิเคราะห์อนุกรมเวลาคือ **ดูกราฟ** ก่อนเลือกโมเดล จุดสีดำแทนเดือนที่จัดโปรโมชั่น

In [ ]:
# @title ▶ กราฟยอดขายรายเดือน
fig, ax = plt.subplots()
ax.plot(df.index, df["sales"], color="black")
ax.scatter(df.index[df.promo == 1], df.sales[df.promo == 1], color="black", s=40, zorder=3, label="เดือนที่จัดโปรโมชั่น")
ax.set_ylabel("ยอดขาย (พันแก้ว)"); ax.set_title(f"Bean There {BRANCH} — ยอดขายรายเดือน ปี 2020–2025"); ax.legend()
plt.show()

In [ ]:
# @title ▶ แยกองค์ประกอบ (decomposition): แนวโน้ม · ฤดูกาล · ส่วนที่เหลือ
dec = seasonal_decompose(df["sales"], model="additive", period=12)
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
parts = [(dec.observed, "ข้อมูลจริง"), (dec.trend, "แนวโน้ม (trend)"),
         (dec.seasonal, "ฤดูกาล (seasonal)"), (dec.resid, "ส่วนที่เหลือ (residual)")]
for ax, (ser, title) in zip(axes, parts):
    ax.plot(ser.index, ser, color="black"); ax.set_title(title, loc="left")
plt.tight_layout(); plt.show()

avg = df.groupby(df.index.month)["sales"].mean().round(1)
avg.index = TH_MONTHS
print("ยอดขายเฉลี่ยแต่ละเดือน (พันแก้ว):"); print(avg.to_string())
print("\nยอดขายเฉลี่ย เดือนมีโปรฯ   :", round(df[df.promo==1].sales.mean(), 1))
print("ยอดขายเฉลี่ย เดือนไม่มีโปรฯ :", round(df[df.promo==0].sales.mean(), 1))

### 📝 คำถามกลุ่ม ข้อ 1–3
1. ยอดขายมี **แนวโน้ม (trend)** ขึ้น ลง หรือคงที่?
2. เดือนไหนขายดีที่สุด และเดือนไหนขายน้อยที่สุด? รูปแบบนี้ **ซ้ำทุกกี่เดือน**?
3. เดือนที่จัดโปรโมชั่นดูเหมือนขายได้มากกว่าเดือนปกติไหม? เพราะอะไรการเทียบค่าเฉลี่ยตรง ๆ แบบนี้อาจยังไม่ยุติธรรม?

In [ ]:
# ✏️ พิมพ์คำตอบของกลุ่มที่นี่ (ถ้าไม่ใช้ใบงานกระดาษ)
answer_1 = ""
answer_2 = ""
answer_3 = ""

## ขั้นที่ 2 · ตรวจความนิ่ง (5 นาที)
ARIMA ต้องการข้อมูลที่ **นิ่ง (stationary)** คือค่าเฉลี่ยไม่เลื่อนไปตามเวลา
เราใช้การทดสอบ **ADF**: ถ้า p-value **น้อยกว่า 0.05** แปลว่านิ่ง

✏️ ลองเลือก `d = 0` แล้วรัน จากนั้นเปลี่ยนเป็น `d = 1` แล้วรันใหม่ เทียบกราฟและ p-value

In [ ]:
# @title ✏️ ทดลองหาผลต่าง (differencing)
d = 0  # @param [0, 1, 2] {type:"raw"}

series = df["sales"].copy()
for _ in range(d):
    series = series.diff()
series = series.dropna()

pval = adfuller(series)[1]
fig, ax = plt.subplots()
ax.plot(series.index, series, color="black")
ax.axhline(series.mean(), color="#9a9a9a", ls="--")
ax.set_title(f"อนุกรมหลังหาผลต่าง d = {d}   |   ADF p-value = {pval:.3f}")
plt.show()
print(f"ADF p-value = {pval:.3f} →", "✅ นิ่งแล้ว" if pval < 0.05 else "❌ ยังไม่นิ่ง ลองเพิ่ม d")

### 📝 คำถามกลุ่ม ข้อ 4
4. ที่ `d = 0` และ `d = 1` ได้ p-value เท่าไร? กลุ่มจะเลือก d เท่าไรสำหรับโมเดล และเพราะอะไร?

## ขั้นที่ 3 · แบ่ง train / test ตามเวลา (2 นาที)
เราซ่อนปี 2025 ไว้เป็น **test** และให้โมเดลเรียนจากปี 2020–2024 (**train**) เท่านั้น
แบ่งตามเวลา ไม่สุ่ม เพื่อให้เหมือนการพยากรณ์อนาคตจริง (60 เดือน train : 12 เดือน test ≈ 83 : 17)

In [ ]:
# @title ▶ ดูการแบ่งข้อมูล
fig, ax = plt.subplots()
ax.plot(train.index, train["sales"], color="#9a9a9a", label=f"ข้อมูลฝึก (train): {len(train)} เดือน")
ax.plot(test.index, test["sales"], color="black", lw=2, label=f"ข้อมูลทดสอบ (test): {len(test)} เดือน (ซ่อนจากโมเดล)")
ax.axvline(test.index[0], color="black", ls=":"); ax.legend(); ax.set_title("แบ่งข้อมูลฝึกและข้อมูลทดสอบตามเวลา")
plt.show()

## ขั้นที่ 4 · โมเดล A: ARIMA(p, d, q) (8 นาที)
เริ่มจากโมเดลพื้นฐานที่ไม่รู้จักฤดูกาลและโปรโมชั่น
✏️ ใช้ค่าเริ่มต้น `(1, 1, 1)` ก่อน แล้วลองปรับ p หรือ q ทีละ 1 ค่า

In [ ]:
# @title ✏️ โมเดล A — ARIMA
p = 1  # @param {type:"slider", min:0, max:3, step:1}
d = 1  # @param {type:"slider", min:0, max:2, step:1}
q = 1  # @param {type:"slider", min:0, max:3, step:1}

model_A = run_model("โมเดล A", p, d, q)

### 📝 คำถามกลุ่ม ข้อ 5–6
5. MAPE ของโมเดล A เท่าไร? อธิบายเป็นภาษาธุรกิจ เช่น “พลาดเฉลี่ยเดือนละกี่แก้ว”
6. ดูกราฟ: เส้นพยากรณ์ (เส้นประ) จับ **ยอดพุ่งเดือนธันวาคม** ได้ไหม? ทำไม?

## ขั้นที่ 5 · โมเดล B: SARIMA (8 นาที)
เพิ่มส่วน **ฤดูกาล** `(P, D, Q, s)` ข้อมูลรายเดือนที่ซ้ำทุกปีใช้ `s = 12`
✏️ เริ่มที่ `(1,1,1) × (1,1,1,12)` แล้วรัน

In [ ]:
# @title ✏️ โมเดล B — SARIMA
p = 1  # @param {type:"slider", min:0, max:3, step:1}
d = 1  # @param {type:"slider", min:0, max:2, step:1}
q = 1  # @param {type:"slider", min:0, max:3, step:1}
P = 1  # @param {type:"slider", min:0, max:2, step:1}
D = 1  # @param {type:"slider", min:0, max:1, step:1}
Q = 1  # @param {type:"slider", min:0, max:2, step:1}
s = 12 # @param [4, 6, 12] {type:"raw"}

model_B = run_model("โมเดล B", p, d, q, P, D, Q, s)

### 📝 คำถามกลุ่ม ข้อ 7–8
7. MAPE ลดลงจากโมเดล A กี่เปอร์เซ็นต์พอยต์? เส้นพยากรณ์เปลี่ยนรูปร่างอย่างไร?
8. ลองเปลี่ยน `s` เป็น 6 หรือ 4 แล้วรัน ผลแย่ลงหรือดีขึ้น? บอกอะไรเราเกี่ยวกับรอบฤดูกาลของร้านนี้?

## ขั้นที่ 6 · โมเดล C: SARIMAX + โปรโมชั่น (8 นาที)
เพิ่ม **ตัวแปรภายนอก** คือ `promo` (1 = เดือนที่จัดโปรฯ) ร้านรู้ปฏิทินโปรฯ ล่วงหน้า จึงใช้พยากรณ์อนาคตได้
✏️ ติ๊ก `use_promo` แล้วรัน

In [ ]:
# @title ✏️ โมเดล C — SARIMAX
p = 1  # @param {type:"slider", min:0, max:3, step:1}
d = 1  # @param {type:"slider", min:0, max:2, step:1}
q = 1  # @param {type:"slider", min:0, max:3, step:1}
P = 1  # @param {type:"slider", min:0, max:2, step:1}
D = 1  # @param {type:"slider", min:0, max:1, step:1}
Q = 1  # @param {type:"slider", min:0, max:2, step:1}
s = 12 # @param [4, 6, 12] {type:"raw"}
use_promo = True  # @param {type:"boolean"}

model_C = run_model("โมเดล C", p, d, q, P, D, Q, s, use_promo)
if model_C is not None and use_promo:
    b = model_C.params["promo"]; pv = model_C.pvalues["promo"]
    print(f"\nค่าสัมประสิทธิ์ promo = {b:.2f}  (p-value = {pv:.3f})")
    print(f"→ เดือนที่จัดโปรฯ ขายเพิ่มขึ้นประมาณ {b*1000:,.0f} แก้ว เมื่อคุมแนวโน้มและฤดูกาลแล้ว")

### 📝 คำถามกลุ่ม ข้อ 9–10
9. MAPE ของโมเดล C เท่าไร? เทียบกับ A และ B
10. ค่าสัมประสิทธิ์ `promo` เท่าไร? แปลเป็นภาษาผู้จัดการว่าอะไร? ถ้าโปรฯ หนึ่งครั้งมีต้นทุนเท่ากับยอดขาย 5,000 แก้ว คุ้มไหม?

## ขั้นที่ 7 · ชาเลนจ์: แข่งลด MAPE (8 นาที)
✏️ ปรับค่าได้อิสระ รันกี่ครั้งก็ได้ ทุกครั้งจะถูกบันทึกในตารางคะแนน
กติกา: **MAPE ต่ำสุดชนะ** เขียนโมเดลที่ดีที่สุดของกลุ่มขึ้นกระดาน

In [ ]:
# @title ✏️ ลองโมเดลของกลุ่ม
p = 0  # @param {type:"slider", min:0, max:3, step:1}
d = 1  # @param {type:"slider", min:0, max:2, step:1}
q = 1  # @param {type:"slider", min:0, max:3, step:1}
P = 0  # @param {type:"slider", min:0, max:2, step:1}
D = 1  # @param {type:"slider", min:0, max:1, step:1}
Q = 1  # @param {type:"slider", min:0, max:2, step:1}
s = 12 # @param [4, 6, 12] {type:"raw"}
use_promo = True  # @param {type:"boolean"}

_ = run_model("ชาเลนจ์", p, d, q, P, D, Q, s, use_promo)

In [ ]:
# @title ▶ ตารางคะแนนของกลุ่ม (เรียงจาก MAPE ต่ำสุด)
leaderboard()

### 📝 คำถามกลุ่ม ข้อ 11
11. โมเดลที่ดีที่สุดของกลุ่มคืออะไร (เขียนค่า p, d, q, P, D, Q, s และ promo)? โมเดลที่ซับซ้อนกว่า (ค่า p, q สูง ๆ) ดีกว่าเสมอไหม?

## ขั้นที่ 8 · พยากรณ์ปี 2026 และนำเสนอ (8 นาที)
นำโมเดลที่ดีที่สุดมา **เรียนจากข้อมูลทั้งหมด 6 ปี** แล้วพยากรณ์ 12 เดือนข้างหน้า
ฝ่ายการตลาดวางแผนจัดโปรฯ ปี 2026 ไว้แล้ว ✏️ เลือกเดือนที่จัดโปรฯ ได้ด้านล่าง

In [ ]:
# @title ✏️ พยากรณ์ปี 2026 ด้วยโมเดลที่ดีที่สุดของกลุ่ม
p = 1  # @param {type:"slider", min:0, max:3, step:1}
d = 1  # @param {type:"slider", min:0, max:2, step:1}
q = 1  # @param {type:"slider", min:0, max:3, step:1}
P = 1  # @param {type:"slider", min:0, max:2, step:1}
D = 1  # @param {type:"slider", min:0, max:1, step:1}
Q = 1  # @param {type:"slider", min:0, max:2, step:1}
s = 12 # @param [4, 6, 12] {type:"raw"}
promo_months_2026 = "4, 7, 12"  # @param {type:"string"}

months = [int(m) for m in promo_months_2026.replace(" ", "").split(",") if m]
future_idx = pd.date_range("2026-01-01", periods=12, freq="MS")
future_promo = pd.DataFrame({"promo": [1 if dt.month in months else 0 for dt in future_idx]}, index=future_idx)

final = SARIMAX(df["sales"], exog=df[["promo"]], order=(p, d, q), seasonal_order=(P, D, Q, s)).fit(disp=False)
fc = final.get_forecast(12, exog=future_promo)
pred, ci = fc.predicted_mean, fc.conf_int()

fig, ax = plt.subplots()
ax.plot(df.index, df["sales"], color="black", label="ค่าจริง 2020–2025")
ax.plot(pred.index, pred, color="black", ls="--", lw=2, label="ค่าพยากรณ์ 2026")
ax.fill_between(ci.index, ci.iloc[:, 0], ci.iloc[:, 1], color="#cccccc", alpha=.6, label="ช่วงความเชื่อมั่น 95%")
ax.set_ylabel("ยอดขาย (พันแก้ว)"); ax.set_title(f"Bean There {BRANCH} — พยากรณ์ยอดขายปี 2026"); ax.legend(loc="upper left")
plt.show()

table = pd.DataFrame({"โปรโมชั่น (1 = มี)": future_promo["promo"].values,
                      "ค่าพยากรณ์ (พันแก้ว)": pred.round(1).values,
                      "ต่ำสุด (95%)": ci.iloc[:, 0].round(1).values,
                      "สูงสุด (95%)": ci.iloc[:, 1].round(1).values},
                     index=[th_month(x) for x in future_idx])
table.index.name = "เดือน"
table

### 📝 คำถามกลุ่ม ข้อ 12 (คำถามผู้จัดการ)
12. **ธันวาคม 2026 ควรเตรียมวัตถุดิบสำหรับกี่แก้ว?** ควรใช้ค่าพยากรณ์กลาง หรือค่าบนของช่วง 95%? ให้เหตุผลเชิงธุรกิจ (ต้นทุนของการขาดสต็อก vs. ของเหลือทิ้ง)

### 🎤 นำเสนอ 1 นาทีต่อกลุ่ม
บอก (1) โมเดลที่ดีที่สุดและ MAPE (2) ผลของโปรโมชั่น (3) ตัวเลขที่แนะนำสำหรับธันวาคม 2026 พร้อมเหตุผล

---
### สรุปสิ่งที่ได้เรียนรู้
- **ดูกราฟก่อนเสมอ**: แนวโน้มบอกค่า d ฤดูกาลบอกค่า s และ D
- **ARIMA → SARIMA → SARIMAX**: เพิ่มฤดูกาล แล้วเพิ่มปัจจัยภายนอก ความแม่นดีขึ้นเมื่อโมเดลตรงกับลักษณะข้อมูล
- **ทดสอบกับอนาคตที่ซ่อนไว้** และอธิบายผลเป็นภาษาธุรกิจ (แก้ว เปอร์เซ็นต์ ต้นทุน)

*ข้อมูลในแบบฝึกหัดนี้เป็นข้อมูลจำลองเพื่อการเรียนการสอน*